# Skill loading, studied

How Claude Code loads a skill, reproduced here with `bro_skills/`:

1. **Discovery** — scan every `bro_skills/*/SKILL.md`, parse only the YAML frontmatter (`name`, `description`). This is cheap and happens for *all* skills up front — like the skill listing Claude sees in its system prompt.
2. **Selection** — pick a skill by matching intent against the descriptions (here: just by name, to keep it simple).
3. **Load** — read the *full* SKILL.md body only for the chosen skill. This is the "progressive disclosure" part: full instructions are loaded lazily, not for every skill.
4. **Use** — feed the loaded instructions to an LLM call (or in this notebook, just print them, since no LLM call is wired up yet).

In [8]:
from pathlib import Path

SKILLS_DIR = Path.cwd().parent / "bro_skills"


def parse_skill_md(path: Path) -> dict:
    """Split a SKILL.md into its YAML frontmatter (as a flat dict) and body.

    Frontmatter here is intentionally simple -- just `key: value` lines
    between two `---` fences -- so no yaml dependency is needed.
    """
    text = path.read_text(encoding="utf-8")
    _, frontmatter, body = text.split("---", 2)

    meta = {}
    for line in frontmatter.strip().splitlines():
        key, _, value = line.partition(":")
        meta[key.strip()] = value.strip()

    return {**meta, "body": body.strip(), "path": path}


def discover_skills(skills_dir: Path) -> dict:
    """Stage 1: cheap discovery. Only name + description, for every skill."""
    registry = {}
    for skill_md in sorted(skills_dir.glob("*/SKILL.md")):
        skill = parse_skill_md(skill_md)
        registry[skill["name"]] = {"description": skill["description"], "path": skill["path"]}
    return registry


skill_registry = discover_skills(SKILLS_DIR)
skill_registry

{'grep-file': {'description': 'Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.',
  'path': WindowsPath('d:/study-on-agent/bro_skills/grep-file/SKILL.md')},
 'hello-world': {'description': 'A minimal example skill. Greets a person by name and explains what it just did. Use this to learn how skill discovery and loading works.',
  'path': WindowsPath('d:/study-on-agent/bro_skills/hello-world/SKILL.md')}}

## Stage 2 — load a chosen skill's full instructions

`skill_registry` only has name + description. Loading the body happens
only for the one skill actually picked -- that's the lazy part.

In [9]:
def load_skill(name: str) -> str:
    """Stage 2: full-body load, only for the requested skill."""
    entry = skill_registry[name]
    skill = parse_skill_md(entry["path"])
    return skill["body"]


instructions = load_skill("hello-world")
print(instructions)

# Hello World

When invoked, do the following:

1. Greet the person by name, warmly and briefly.
2. In one sentence, explain that you were invoked as the `hello-world` skill.

Keep the whole response to two short sentences. Do not add anything else.


## Interactive skills — tools the kernel can actually run

A skill is instructions, not code. `grep-file/SKILL.md` tells the agent to
call `grep_file(pattern, path)` — here are those tools, and re-running
discovery now that a second skill exists.

In [10]:
REPO_ROOT = SKILLS_DIR.parent


def _resolve(path: str) -> Path:
    p = Path(path)
    return p if p.is_absolute() else REPO_ROOT / p


def read_file(path: str) -> str:
    return _resolve(path).read_text(encoding="utf-8")


def grep_file(pattern: str, path: str) -> list[tuple[int, str]]:
    return [
        (i, line)
        for i, line in enumerate(_resolve(path).read_text(encoding="utf-8").splitlines(), start=1)
        if pattern in line
    ]


TOOLS = {"read_file": read_file, "grep_file": grep_file}

# re-discover now that bro_skills/ has a second skill
skill_registry = discover_skills(SKILLS_DIR)
skill_registry

{'grep-file': {'description': 'Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.',
  'path': WindowsPath('d:/study-on-agent/bro_skills/grep-file/SKILL.md')},
 'hello-world': {'description': 'A minimal example skill. Greets a person by name and explains what it just did. Use this to learn how skill discovery and loading works.',
  'path': WindowsPath('d:/study-on-agent/bro_skills/hello-world/SKILL.md')}}

## Routing — letting a model pick the skill

This is the part Claude Code does with a real LLM call: hand the model
every `(name, description)` pair plus the user's request, and let it
choose. Wired here through `brollm.BaseContract` to Bedrock's `converse`
API, model `google.gemma-3-4b-it` in `us-east-1`.

In [11]:
import boto3
from brollm import BaseContract

BEDROCK_REGION = "us-east-1"
BEDROCK_MODEL_ID = "google.gemma-3-4b-it"

bedrock_client = boto3.client("bedrock-runtime", region_name=BEDROCK_REGION)


def build_router_prompt(user_request: str, registry: dict) -> str:
    listing = "\n".join(f"- {name}: {info['description']}" for name, info in registry.items())
    return (
        "Given this user request, pick the single best-matching skill by name.\n"
        "Reply with only the skill name, nothing else.\n\n"
        f"Skills:\n{listing}\n\n"
        f"User request: {user_request}"
    )


def input_fn(user_request: str, registry: dict):
    prompt = build_router_prompt(user_request, registry)
    return bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
    )


def output_fn(response) -> str:
    text = response["output"]["message"]["content"][0]["text"]
    return text.strip().strip("`").strip()


router = BaseContract(input_fn=input_fn, output_fn=output_fn)

chosen = router(user_request="find every place we log errors in dev.ipynb", registry=skill_registry)
print(chosen)
instructions = load_skill(chosen)
print(instructions)

grep-file
# Grep File

When invoked, you have access to two tools: `read_file(path)` and
`grep_file(pattern, path)`.

1. Identify the file path and the pattern from the request.
2. Call `grep_file(pattern, path)`.
3. Report each match as `line_number: line_text`. If there are no matches,
   say so plainly.

Do not read the whole file unless the user also asks for that.


In [12]:
r_prompt = build_router_prompt("find every place we log errors in dev.ipynb", skill_registry)
print(r_prompt)

Given this user request, pick the single best-matching skill by name.
Reply with only the skill name, nothing else.

Skills:
- grep-file: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.
- hello-world: A minimal example skill. Greets a person by name and explains what it just did. Use this to learn how skill discovery and loading works.

User request: find every place we log errors in dev.ipynb


## Executing the chosen skill

Routing only produces a name. Executing means: ask the model, given the
skill's instructions, to decide which tool to call and with what
arguments; run that tool for real in the kernel; then feed the result
back so the model can write the final answer following the skill's
instructions. Three steps, so this is a `broflow` `Flow` — each step
decides its own next step, exactly like `PlanTask` here choosing `act`
when a tool is needed and `respond` when it isn't (e.g. `hello-world`
needs no tool at all).

In [13]:
import json
from brollm import extract_codeblocks
from broflow import BaseTask, TaskRegistry, Flow


def call_bedrock(prompt: str) -> str:
    response = bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
    )
    return response["output"]["message"]["content"][0]["text"]


def build_plan_prompt(instructions: str, user_request: str) -> str:
    tool_docs = "\n".join(f"- {name}" for name in TOOLS)
    return (
        f"{instructions}\n\n"
        f"Available tools:\n{tool_docs}\n\n"
        "Decide which tool call satisfies the user's request, if any.\n"
        "Reply with only a fenced json codeblock, either:\n"
        '```json\n{"tool": "grep_file", "args": {"pattern": "...", "path": "..."}}\n```\n'
        "or, if no tool is needed:\n"
        '```json\n{"tool": null}\n```\n\n'
        f"User request: {user_request}"
    )


class PlanTask(BaseTask):
    possible_next = {"act", "respond"}

    def __call__(self, state, **kwargs):
        prompt = build_plan_prompt(state["instructions"], state["user_request"])
        raw = call_bedrock(prompt)
        json_blocks = [b for b in extract_codeblocks(raw) if b.language == "json"]
        call = json.loads(json_blocks[0].content)
        state["tool"] = call.get("tool")
        state["args"] = call.get("args", {})
        self.set_next("act" if state["tool"] else "respond")
        return state


class ActTask(BaseTask):
    possible_next = {"respond"}

    def __call__(self, state, **kwargs):
        state["tool_result"] = TOOLS[state["tool"]](**state["args"])
        self.set_next("respond")
        return state


class RespondTask(BaseTask):
    possible_next = {"end"}

    def __call__(self, state, **kwargs):
        prompt = (
            f"{state['instructions']}\n\n"
            f"User request: {state['user_request']}\n"
        )
        if state.get("tool"):
            prompt += f"Tool `{state['tool']}` returned: {state['tool_result']}\n\n"
        prompt += "Write the final answer to the user, following the skill's instructions."
        state["answer"] = call_bedrock(prompt)
        self.set_next("end")
        return state


registry = TaskRegistry()
registry.register("plan", PlanTask("plan"))
registry.register("act", ActTask("act"))
registry.register("respond", RespondTask("respond"))
skill_flow = Flow(registry)


def execute_skill(skill_name: str, user_request: str) -> dict:
    state = {"instructions": load_skill(skill_name), "user_request": user_request}
    skill_flow.run(start="plan", end="end", state=state)
    return state

In [14]:
user_request = "grep for the word 'description' in bro_skills/grep-file/SKILL.md"

chosen = router(user_request=user_request, registry=skill_registry)
print("routed to:", chosen)

result = execute_skill(chosen, user_request)
print(skill_flow.trace)
print(result["answer"])

routed to: grep-file
[('plan', 'act'), ('act', 'respond'), ('respond', 'end')]
line_1: 3: description: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.
